# Notebook 3 — Joint-Schedule FT on UICD (Colab A100)

**Purpose:** Mahapatra's highest-priority new baseline. Same two-phase
learning-rate schedule as Staged FT (5e-5 for 2 epochs, then 1e-5 for 3
epochs) but with **no freezing** -- all parameters trainable throughout.
Isolates whether Staged FT's behavior comes from the freeze/unfreeze
structure itself, or just from the LR schedule shape.

**Environment pins (must match Kaggle, verified against your RSICD seed-42 rerun):**
- transformers == 4.41.2
- peft == 0.11.1

**Data source:** UICD images + captions come from a Kaggle Dataset
(`kiranmuhammad/uicd-underwater-dataset`), not HuggingFace. This notebook
downloads it via the Kaggle API before training.

**Runtime:** UICD is small (~2.2K train images) -- expect ~1-2 hours per
seed on A100, ~4-6 hours total for 3 seeds.

**Output:** `joint_schedule_ft_uicd_seed{42,0,123}.json` in
`/content/drive/MyDrive/DAMF/logs/`, matching the schema of your existing
experiment logs (same `rt_log` format, same metric fields) so Notebook 1's
analysis code works on it unmodified.

**Run order:** top to bottom. Restart runtime once after Cell 2 installs
the pinned transformers version (Colab preloads a newer one that must be
displaced).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install pinned dependencies (matches Kaggle exactly)

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                    '-q', '--no-warn-script-location'], check=True)

pip_install('transformers==4.41.2')
pip_install('peft==0.11.1')
pip_install('pycocoevalcap')
pip_install('nltk')
pip_install('datasets')
pip_install('evaluate')
pip_install('kaggle')

print('Installations complete. RESTART RUNTIME NOW (Runtime -> Restart runtime),')
print('then re-run cells 1 and 2 (skip this print block) before proceeding to cell 3.')
print('Reason: Colab preloads a newer transformers; the pinned 4.41.2 will not')
print('take effect until runtime restart.')

## IMPORTANT -- restart runtime once

After Cell 2 completes, click **Runtime -> Restart runtime**, then re-run cells 1 and 2. Verify with the next cell before continuing.

In [ ]:
import transformers, peft
print(f'transformers : {transformers.__version__}  (need 4.41.2)')
print(f'peft         : {peft.__version__}  (need 0.11.1)')
assert transformers.__version__ == '4.41.2', \
    f'STOP: transformers is {transformers.__version__}. Restart runtime and re-run cell 2.'
assert peft.__version__ == '0.11.1', \
    f'STOP: peft is {peft.__version__}. Restart runtime and re-run cell 2.'
print('Versions verified.')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
print('NLTK data downloaded.')

## 3. Download UICD dataset from Kaggle

Requires a `kaggle.json` API token. Upload yours to
`/content/drive/MyDrive/kaggle.json` before running this cell (see
kaggle.com/settings -> API -> Legacy API Credentials -> Create New Token
if you don't already have one saved).

In [ ]:
import os, shutil
from pathlib import Path

kaggle_json_src = Path('/content/drive/MyDrive/kaggle.json')
assert kaggle_json_src.exists(), (
    'kaggle.json not found in Drive. Go to kaggle.com/settings, under '
    '"Legacy API Credentials" click Create New Token, and upload the '
    'downloaded kaggle.json to /content/drive/MyDrive/kaggle.json'
)

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(kaggle_json_src, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

uicd_dir = Path('/content/uicd_data')
if not uicd_dir.exists() or not any(uicd_dir.iterdir()):
    uicd_dir.mkdir(exist_ok=True)
    print('Downloading UICD dataset from Kaggle (first run only)...')
    subprocess.run(
        ['kaggle', 'datasets', 'download',
         '-d', 'kiranmuhammad/uicd-underwater-dataset',
         '-p', str(uicd_dir), '--unzip'],
        check=True
    )
else:
    print('UICD data already present in session storage.')

found = list(uicd_dir.rglob('UIC-captions.txt'))
print(f'\nFound {len(found)} UIC-captions.txt file(s):')
for f in found:
    print(f'  {f}')

In [ ]:
captions_files = list(Path('/content/uicd_data').rglob('UIC-captions.txt'))
assert captions_files, (
    'UIC-captions.txt not found after download. Check the Kaggle dataset '
    'slug is correct, or inspect /content/uicd_data manually.'
)
UICD_CAPS = str(captions_files[0])
UICD_BASE = str(captions_files[0].parent)

image_dirs = [d for d in Path(UICD_BASE).iterdir()
             if d.is_dir() and 'image' in d.name.lower()]
assert image_dirs, f'No image directory found under {UICD_BASE}'
UICD_IMAGES = str(image_dirs[0])

print(f'UICD_CAPS   = {UICD_CAPS}')
print(f'UICD_IMAGES = {UICD_IMAGES}')
n_images = len(list(Path(UICD_IMAGES).glob('*')))
print(f'Images found: {n_images} (expected ~3176)')

## 4. Imports and environment report

In [ ]:
import sys, gc, json, math, random
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from collections import defaultdict

from transformers import BlipProcessor, BlipForConditionalGeneration
from peft import LoraConfig, get_peft_model
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

print('=' * 55)
print('ENVIRONMENT REPORT')
print('=' * 55)
print(f'Python       : {sys.version.split()[0]}')
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')
print(f'PEFT         : {peft.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU          : {props.name}  ({props.total_memory/1e9:.1f} GB)')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Active device: {DEVICE}')

## 5. Configuration (verbatim from Kaggle Cell 37)

In [ ]:
OUT = '/content/drive/MyDrive/DAMF/logs'
os.makedirs(OUT, exist_ok=True)

SEEDS = [42, 0, 123]
SPLIT_SEED = 42

CFG = {
    'train_ratio': 0.70, 'val_ratio': 0.15, 'test_ratio': 0.15,
    'batch_size': 16, 'max_length': 30, 'beam_size': 3, 'num_workers': 2,
    'stage1_epochs': 2, 'stage2_epochs': 3, 'total_naive_epochs': 5,
    'lr_naive': 1e-4, 'lr_lowlr': 1e-5, 'lr_stage1': 5e-5, 'lr_stage2': 1e-5,
    'lr_lora': 1e-4, 'weight_decay': 0.01,
    'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'rt_log_every_n_steps': 10, 'rt_epsilon': 1e-8,
}

print(f'CFG loaded. Output directory: {OUT}')

## 6. GradientTracker (verbatim from Kaggle Cell 9)

In [ ]:
class GradientTracker:
    def __init__(self, model, model_type='blip'):
        self.eps = CFG['rt_epsilon']
        self.model_type = model_type
        self._vis_norms = []
        self._lang_norms = []
        self._hooks = []

        if model_type == 'blip':
            vis_key, lang_key = 'vision_model', 'text_decoder'
        elif model_type == 'blip2':
            vis_key, lang_key = 'vision_model', 'language_model'
        else:
            raise ValueError(f'Unknown model_type: {model_type}')

        n_vis, n_lang, n_skipped = 0, 0, 0
        for name, param in model.named_parameters():
            if not param.requires_grad:
                n_skipped += 1
                continue
            if vis_key in name:
                def make_vis_hook(n=name):
                    def hook(grad):
                        if grad is not None:
                            self._vis_norms.append(grad.detach().norm().item())
                    return hook
                self._hooks.append(param.register_hook(make_vis_hook()))
                n_vis += 1
            elif lang_key in name:
                def make_lang_hook(n=name):
                    def hook(grad):
                        if grad is not None:
                            self._lang_norms.append(grad.detach().norm().item())
                    return hook
                self._hooks.append(param.register_hook(make_lang_hook()))
                n_lang += 1

        print(f'  GradientTracker hooked: visual={n_vis}, language={n_lang}, frozen={n_skipped}')
        if n_vis == 0:
            print('  WARNING: No visual parameters hooked.')
        if n_lang == 0:
            print('  WARNING: No language parameters hooked.')

    def get_rt(self):
        if not self._vis_norms or not self._lang_norms:
            return None
        nv = math.sqrt(sum(v ** 2 for v in self._vis_norms))
        nl = math.sqrt(sum(v ** 2 for v in self._lang_norms))
        return nl / (nv + self.eps)

    def reset(self):
        self._vis_norms.clear()
        self._lang_norms.clear()

    def remove(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()
        print('  GradientTracker hooks removed.')

print('GradientTracker defined.')

## 7. Training utilities (verbatim from Kaggle Cell 10)

In [ ]:
def compute_bleu4(predictions, references):
    smoother = SmoothingFunction().method4
    pred_tokens = [p.split() for p in predictions]
    ref_tokens = [[r.split() for r in refs] for refs in references]
    return corpus_bleu(ref_tokens, pred_tokens, smoothing_function=smoother)

def compute_cider(predictions, references):
    try:
        from pycocoevalcap.cider.cider import Cider
        from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
        gts = {i: [{'caption': r} for r in refs] for i, refs in enumerate(references)}
        res = {i: [{'caption': p}] for i, p in enumerate(predictions)}
        tok = PTBTokenizer()
        sc, _ = Cider().compute_score(tok.tokenize(gts), tok.tokenize(res))
        return float(sc)
    except Exception as e:
        print(f'  CIDEr error: {e}')
        return None

def compute_meteor(predictions, references):
    try:
        import evaluate as hf_evaluate
        meteor = hf_evaluate.load('meteor')
        flat_refs = [refs[0] for refs in references]
        result = meteor.compute(predictions=predictions, references=flat_refs)
        return float(result['meteor'])
    except Exception as e:
        print(f'  METEOR error: {e}')
        return None

@torch.no_grad()
def evaluate_blip(model, loader, processor, image_captions, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(
            pixel_values=batch['pixel_values'].to(DEVICE),
            max_length=cfg['max_length'], num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        for img_name in batch['image_name']:
            references.append(image_captions[img_name])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

def get_val_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            inputs = {
                'pixel_values': batch['pixel_values'].to(DEVICE),
                'input_ids': batch['input_ids'].to(DEVICE),
                'attention_mask': batch['attention_mask'].to(DEVICE),
            }
            inputs['labels'] = inputs['input_ids'].clone()
            with torch.amp.autocast('cuda'):
                total += model(**inputs).loss.item()
            n += 1
    return total / max(n, 1)

def train_one_epoch(model, loader, optimizer, scaler, tracker=None, step_counter=None):
    model.train()
    total_loss, n_batches, rt_log = 0.0, 0, []
    if step_counter is None:
        step_counter = [0]
    for batch in loader:
        inputs = {
            'pixel_values': batch['pixel_values'].to(DEVICE),
            'input_ids': batch['input_ids'].to(DEVICE),
            'attention_mask': batch['attention_mask'].to(DEVICE),
        }
        inputs['labels'] = inputs['input_ids'].clone()
        optimizer.zero_grad()
        if tracker is not None:
            tracker.reset()
        with torch.amp.autocast('cuda'):
            loss = model(**inputs).loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches += 1
        step_counter[0] += 1
        if tracker is not None and step_counter[0] % CFG['rt_log_every_n_steps'] == 0:
            rt = tracker.get_rt()
            if rt and not math.isinf(rt) and not math.isnan(rt):
                rt_log.append((step_counter[0], rt))
    return total_loss / max(n_batches, 1), rt_log

def save_logs(logs, filename):
    path = os.path.join(OUT, filename)
    with open(path, 'w') as f:
        json.dump(logs, f, indent=2)
    print(f'  Saved: {filename}')
    return path

print('Utilities defined.')

## 8. UICD dataset and data loading (verbatim pattern from Kaggle Cell 37)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_uicd_captions(captions_path):
    image_captions = defaultdict(list)
    with open(captions_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('#')
            if len(parts) < 2:
                continue
            img_name = parts[0].strip()
            cap_part = parts[1].strip()
            caption = cap_part.split(' ', 1)[1].strip() if ' ' in cap_part else cap_part
            if img_name and caption:
                image_captions[img_name].append(caption)
    return dict(image_captions)

image_captions = load_uicd_captions(UICD_CAPS)
_split_rng = random.Random(SPLIT_SEED)
_all_images = sorted(image_captions.keys())
_split_rng.shuffle(_all_images)
n = len(_all_images)
train_end = int(CFG['train_ratio'] * n)
val_end = int((CFG['train_ratio'] + CFG['val_ratio']) * n)
SPLITS = {
    'train': _all_images[:train_end],
    'val': _all_images[train_end:val_end],
    'test': _all_images[val_end:],
}
print(f"UICD split (fixed, seed={SPLIT_SEED}) -- "
      f"train:{len(SPLITS['train'])} val:{len(SPLITS['val'])} test:{len(SPLITS['test'])}")
print('IMPORTANT: verify these counts match your paper (2223/476/477).')

BLIP_PROCESSOR = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')

class UICDataset(Dataset):
    def __init__(self, image_list, image_captions, image_folder, processor,
                max_length, deterministic=False):
        self.image_list = image_list
        self.image_captions = image_captions
        self.image_folder = image_folder
        self.processor = processor
        self.max_length = max_length
        self.deterministic = deterministic

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        image = Image.open(os.path.join(self.image_folder, img_name)).convert('RGB')
        caption = (self.image_captions[img_name][0] if self.deterministic
                  else random.choice(self.image_captions[img_name]))
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'image_name': img_name,
        }

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loaders():
    g = torch.Generator()
    g.manual_seed(0)
    train_loader = DataLoader(
        UICDataset(SPLITS['train'], image_captions, UICD_IMAGES, BLIP_PROCESSOR,
                  CFG['max_length'], deterministic=False),
        batch_size=CFG['batch_size'], shuffle=True,
        num_workers=CFG['num_workers'], pin_memory=True,
        worker_init_fn=worker_init_fn, generator=g)
    val_loader = DataLoader(
        UICDataset(SPLITS['val'], image_captions, UICD_IMAGES, BLIP_PROCESSOR,
                  CFG['max_length'], deterministic=True),
        batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], pin_memory=True)
    return train_loader, val_loader

train_loader, val_loader = make_loaders()
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

sb = next(iter(val_loader))
assert sb['pixel_values'].shape[1:] == torch.Size([3, 224, 224]), \
    f"Unexpected shape: {sb['pixel_values'].shape}"
print('Smoke test passed.')

## 9. NEW: Joint-Schedule FT runner

Same disk-safety helpers as your Kaggle infrastructure (`check_disk_space`, resume-safe JSON checkpointing).

In [ ]:
import shutil as _shutil

def check_disk_space(min_gb=3.0):
    free_gb = _shutil.disk_usage(OUT).free / 1e9
    if free_gb < min_gb:
        print(f'  !! DISK WARNING: only {free_gb:.2f} GB free.')
        return False
    return True

def save_checkpoint(model, filename):
    if not check_disk_space():
        return None
    path = os.path.join(OUT, filename)
    try:
        torch.save(model.state_dict(), path)
        return path
    except RuntimeError as e:
        print(f'  !! Checkpoint save failed ({filename}): {e}')
        return None

def delete_checkpoint(filename):
    path = os.path.join(OUT, filename)
    if os.path.exists(path):
        os.remove(path)

def run_uicd_joint_schedule(seed):
    """Joint-Schedule FT: identical LR schedule to Staged FT
    (5e-5 for 2 epochs, then 1e-5 for 3 epochs) but with ALL
    parameters trainable throughout -- no freezing at any point.
    Rt is well-defined and logged for the FULL 5 epochs, unlike
    Staged FT where Stage 1 has an undefined Rt (decoder frozen).
    """
    exp_name = f'joint_schedule_ft_uicd_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')
    total_epochs = CFG['stage1_epochs'] + CFG['stage2_epochs']

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= total_epochs:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
        print(f'  [{exp_name}] resuming from epoch {done + 1}')
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'UICD', 'seed': seed,
            'method': 'joint_schedule_ft',
            'phase1_lr': CFG['lr_stage1'], 'phase1_epochs': CFG['stage1_epochs'],
            'phase2_lr': CFG['lr_stage2'], 'phase2_epochs': CFG['stage2_epochs'],
            'freeze_vision': False, 'freeze_language': False,
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [], 'rt_log': [],
        }
        done = 0

    seed_everything(seed)
    model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base').to(DEVICE)

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0
    scaler = torch.amp.GradScaler('cuda')
    tracker = GradientTracker(model, 'blip')
    step_counter = [done * len(train_loader)]

    if done < CFG['stage1_epochs']:
        opt1 = AdamW(model.parameters(), lr=CFG['lr_stage1'],
                    weight_decay=CFG['weight_decay'])
        for epoch in range(done + 1, CFG['stage1_epochs'] + 1):
            avg_loss, rt_log = train_one_epoch(model, train_loader, opt1, scaler,
                                                tracker=tracker, step_counter=step_counter)
            val_loss = get_val_loss(model, val_loader)
            bleu4, cider, meteor, preds, refs = evaluate_blip(
                model, val_loader, BLIP_PROCESSOR, image_captions, CFG)
            logs['train_loss_per_epoch'].append(round(avg_loss, 4))
            logs['val_loss_per_epoch'].append(round(val_loss, 4))
            logs['bleu4_per_epoch'].append(round(bleu4, 4))
            logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
            logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
            logs['rt_log'].extend(rt_log)
            if bleu4 > best_bleu4:
                best_bleu4 = bleu4
                logs['best_bleu4'] = bleu4
                logs['best_cider'] = cider
                logs['best_meteor'] = meteor
                logs['best_epoch'] = epoch
                logs['best_predictions'] = preds[:20]
                logs['best_references'] = [r[:2] for r in refs[:20]]
            cs = f'{cider:.4f}' if cider else 'N/A'
            ms = f'{meteor:.4f}' if meteor else 'N/A'
            print(f"  [{exp_name}] P1 epoch {epoch}/{CFG['stage1_epochs']} | "
                  f'train={avg_loss:.4f} val={val_loss:.4f} '
                  f'BLEU-4={bleu4:.4f} CIDEr={cs} METEOR={ms}')
            save_checkpoint(model, f'{exp_name}_last.pt')
            save_logs(logs, f'{exp_name}.json')
        done = CFG['stage1_epochs']

    opt2 = AdamW(model.parameters(), lr=CFG['lr_stage2'],
                weight_decay=CFG['weight_decay'])
    for epoch in range(max(done, CFG['stage1_epochs']) + 1, total_epochs + 1):
        avg_loss, rt_log = train_one_epoch(model, train_loader, opt2, scaler,
                                            tracker=tracker, step_counter=step_counter)
        val_loss = get_val_loss(model, val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip(
            model, val_loader, BLIP_PROCESSOR, image_captions, CFG)
        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)
        logs['rt_log'].extend(rt_log)
        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = [r[:2] for r in refs[:20]]
        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f'  [{exp_name}] P2 epoch {epoch}/{total_epochs} | '
              f'train={avg_loss:.4f} val={val_loss:.4f} '
              f'BLEU-4={bleu4:.4f} CIDEr={cs} METEOR={ms}')
        save_checkpoint(model, f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')

    tracker.remove()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_uicd_joint_schedule defined.')

## 10. LAUNCH -- all 3 seeds

Resume-safe: if this cell is interrupted, re-running it picks up exactly where each seed left off (per-epoch checkpointing, same pattern as your existing infrastructure).

In [ ]:
print('=' * 60)
print('JOINT-SCHEDULE FT -- UICD, all 3 seeds')
print('=' * 60)

results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    results[seed] = run_uicd_joint_schedule(seed)

print('\n' + '=' * 60)
print('ALL SEEDS COMPLETE')
print('=' * 60)
for seed, logs in results.items():
    rt = logs.get('rt_log', [])
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}  "
          f'rt_log entries={len(rt)}')
    if rt:
        print(f'   first: step {rt[0][0]}, Rt {rt[0][1]:.3f}  '
              f'last: step {rt[-1][0]}, Rt {rt[-1][1]:.3f}')

print(f'\nSaved to {OUT}/joint_schedule_ft_uicd_seed*.json')
print('Next: extend Notebook 1 (METHODS list) to include "joint_schedule_ft"')
print('and re-run its analysis cells to compare against Low-LR FT and Staged FT.')